In [1]:
import numpy as np
import matplotlib
matplotlib.use('TkAgg')  # Use TkAgg backend for interactive features
import matplotlib.pyplot as plt
from scipy.optimize import minimize, LinearConstraint, Bounds
from tkinter import filedialog
import tkinter as tk
import pandas as pd
from pyspectra.readers.read_spc import read_spc
from scipy.signal import savgol_filter
from pyspectra.transformers.spectral_correction import snv

In [2]:
# Read pure form spectra and visualize

files_pure = filedialog.askopenfilenames(
    title='Select one or more pure spectra',
    filetypes=[('All files', '*.*')]
)
path_pure = files_pure[0].rsplit('/', 1)[0] + '/'

K = []
AXIS = None

for file in files_pure:
    temp = read_spc(file)
    K.append(temp.values)
    AXIS = temp.index

K = np.array(K)

# Plot raw pure form spectra

fig1, ax1 = plt.subplots()
for i, file in enumerate(files_pure):
    filename = file.rsplit('/', 1)[-1]
    ax1.plot(AXIS, K[i, :], label=filename)
ax1.set_xlabel('wavenumber (cm-1)')
ax1.set_ylabel('Raman counts')
ax1.set_title('raw pure component spectra')
ax1.legend()
ax1.grid(True)
plt.tight_layout()
plt.show()

# Plot preprocessed pure form spectra

fig2, ax2 = plt.subplots()
for i, file in enumerate(files_pure):
    SNV=snv()
    filename = file.rsplit('/', 1)[-1]
    ax2.plot(AXIS, SNV.fit_transform(savgol_filter(K[i, :], 5, 3, 2)), label=filename) #the parameter of 5,3,2 can be changed if necessary
ax2.set_xlabel('wavenumber (cm-1)')
ax2.set_ylabel('Raman counts')
ax2.set_title('preprocessed pure component spectra')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

x-y(1)
gx-y(1)
x-y(1)


In [3]:
# Read sample spectra and visualize
files = filedialog.askopenfilenames(
    title='Select one or more sample spectra',
    filetypes=[('All files', '*.*')]
)

X = []
for file in files:
    temp = read_spc(file)
    X.append(temp.values)
    AXIS = temp.index

X = np.array(X)

# Plot raw spectra (pure form + samples)
fig3, ax3 = plt.subplots()
for i, file in enumerate(files_pure):
    filename = file.rsplit('/', 1)[-1]
    ax3.plot(AXIS, K[i, :], label=filename)
for j, file in enumerate(files):
    filename = file.rsplit('/', 1)[-1]
    ax3.plot(AXIS, X[j, :] + 0.5 * np.max(K), 'k-', label=filename)
ax3.set_xlabel('wavenumber (cm-1)')
ax3.set_ylabel('Raman counts')
ax3.set_title('raw spectra')
ax3.legend()
ax3.grid(True)
plt.tight_layout()
plt.show()

# Plot preprocessed spectra (pure form + samples)
fig4, ax4 = plt.subplots()
for i, file in enumerate(files_pure):
    filename = file.rsplit('/', 1)[-1]
    SNV=snv()
    ax4.plot(AXIS, SNV.fit_transform(savgol_filter(K[i, :], 5, 3, 2)), label=filename) #the parameter of 5,3,2 can be changed if necessary
for j, file in enumerate(files):
    filename = file.rsplit('/', 1)[-1]
    SNV=snv()
    ax4.plot(AXIS, SNV.fit_transform(savgol_filter(X[j, :], 5, 3, 2)) + 10, 'k-', label=filename) #the parameter of 5,3,2 can be changed if necessary
ax4.set_xlabel('wavenumber (cm-1)')
ax4.set_ylabel('Raman counts')
ax4.set_title('preprocessed spectra')
ax4.legend()
ax4.grid(True)
plt.tight_layout()
plt.show()

gx-y(1)
gx-y(1)
gx-y(1)


In [4]:
#IOT code
import numpy as np
from scipy.optimize import minimize, LinearConstraint, Bounds

def EIOT_pred(S_E, NUM_NC, dm, c_A_bounds, beq):

    # Compute H and f matrices
    H = S_E.T @ S_E
    f = -S_E.T @ dm
    
    A = np.array([]).reshape(0, S_E.shape[1])  # Empty constraint matrix
    b = np.array([])
    
    # Set bounds and equality constraints for non-chemical intereferences (not applicable for polymorphism case)
    if len(c_A_bounds) != 0:
        # Constrained case
        Aeq = np.hstack([np.ones((1, S_E.shape[1] - NUM_NC)), 
                          np.zeros((1, NUM_NC))])
        lb = np.hstack([np.zeros(S_E.shape[1] - NUM_NC), 
                        np.ones(NUM_NC) * c_A_bounds[0]])
        ub = np.hstack([np.ones(S_E.shape[1] - NUM_NC), 
                        np.ones(NUM_NC) * c_A_bounds[1]])
    else:
        # Unconstrained case
        Aeq = np.ones((1, S_E.shape[1]))
        lb = np.zeros(S_E.shape[1])
        ub = np.ones(S_E.shape[1])
    
    # Define objective function: 0.5 * x^T @ H @ x + f @ x
    def objective(x):
        return 0.5 * x @ H @ x + f @ x
    
    # Define equality constraint: Aeq @ x = beq
    constraints = LinearConstraint(Aeq, beq, beq)
    bounds = Bounds(lb, ub)
    
    # Solve quadratic program
    result = minimize(
        objective,
        x0=np.zeros(S_E.shape[1]),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-9, 'maxiter': 1000}
    )
    
    c_E_hat = result.x
    fval = result.fun
    exitflag = result.success
    output = result
    lambda_vals = result
    
    # Compute results
    dm_hat = S_E @ c_E_hat
    Em = dm - dm_hat
    sse = np.sum((dm - dm_hat) ** 2)
    
    return c_E_hat, Em, sse, lambda_vals


In [5]:
# IOT Analysis

mask = (AXIS >= 1511) & (AXIS <= 1733)   #feel free to change the wavenumber range of your interest 
b = np.where(mask)[0]

temp_K=savgol_filter(K,5,3,2)
temp_X=savgol_filter(X,5,3,2)
SNV=snv()
temp_K_1=SNV.fit_transform(temp_K.transpose())  #Input has to be the right orientation before feeding into snv
SNV=snv()
temp_X_1=SNV.fit_transform(temp_X.transpose())  #Input has to be the right orientation before feeding into snv
#fig5, ax5 = plt.subplots()
#ax5.plot(AXIS, temp_K_1,'r-')
#ax5.plot(AXIS,temp_X_1,'b-')
#plt.tight_layout()
#plt.show()

PRED = []

for k in range(len(files)):
    
    c_E_hat, Em, sse, lamda_vals = EIOT_pred(temp_K_1[b, :],0,temp_X_1[b, k],[],1)
    PRED.append(c_E_hat)

PRED = np.array(PRED)

# Create results table
filenames_pure = [f.rsplit('/', 1)[-1] for f in files_pure]
filenames = [f.rsplit('/', 1)[-1] for f in files]

T = pd.DataFrame(PRED, index=filenames, columns=filenames_pure)
print(T)
T.to_csv('RESULTS.csv')


                                          form_I.spc  form_J.spc  \
100642-032_2026-04-30-23.09.29.870.spc  1.030648e-12         1.0   
100642-032_2026-04-30-23.14.33.060.spc  0.000000e+00         1.0   
100642-032_2026-04-30-23.19.36.240.spc  0.000000e+00         1.0   

                                        intermediate form.spc  
100642-032_2026-04-30-23.09.29.870.spc           0.000000e+00  
100642-032_2026-04-30-23.14.33.060.spc           3.576855e-13  
100642-032_2026-04-30-23.19.36.240.spc           0.000000e+00  
